# 전체 데이터셋·FR·PQ의 FIQA + saliency 보정 실험

LFW / RFW-Custom / SurvFace × ArcFace / AdaFace / MagFace / EdgeFace × PQ m128 / m64 / m32의 **36개 조건**을 비교합니다. 기존 `00_batch_experiment_runner.ipynb`의 완료 run을 입력으로 사용합니다.

첫 코드 셀에서 설정하고 **Kernel Restart → Run All** 하세요. 기본 `EXECUTE=False`는 36개 source 조건을 검증합니다. `EXECUTE=True`는 누락 입력 생성 → 공통 CI → 전체 분할 안정성 → 통합 보고를 실행합니다. 완료된 seed별 결과는 hash 검증 후 재사용합니다.

In [1]:
from pathlib import Path
import sys
from copy import deepcopy

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "research").is_dir() and (p / ".git").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from research.experiments.calibration_matrix import DEFAULT_RUN_MATRIX, DEFAULT_SALIENCY_INPUTS

# 전체 36개 조건. 결과에 따라 모델/압축률/seed를 골라 제외하지 않습니다.
DATASETS = ("lfw", "rfw_custom", "survface")
FR_MODELS = ("arcface", "adaface", "magface", "edgeface")
PQ_PROFILES = ("pq_512_m128_b8", "pq_512_m64_b8", "pq_512_m32_b8")
MODEL_RUN_MATRIX = deepcopy(DEFAULT_RUN_MATRIX)  # 명시적으로 고정된 완료 12개 run; 아래 검증 표에서 확인

EXECUTE = True  # True로 변경하면 정식 36조건 실행
RUN_SPLIT_STABILITY = True
FIQA_VARIANT = "L"
FIQA_BATCH_SIZE = 64
PARTITION_SEED = 8972
SPLIT_SEEDS = (*range(19), 8972)
TARGET_FPIRS = (.01, .05, .10, .20, .30)
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 8972
SAFETY_FRACTION = .30
KNOT_QUANTILES = (1/3, 2/3)
SMOOTHING = .01
RIDGE = .001
MAX_ITERATIONS = 2000
MINIMUM_GROUP_NON_MATED = 100
SHRINKAGE_STRENGTH = 200.

RESULT_ROOT = PROJECT_ROOT / "results/calibration"
MATRIX_ROOT = RESULT_ROOT / "matrix"
# 기존 SurvFace m128의 완료 saliency+가림 자료를 명시적으로 재사용합니다.
# 새 모델/run 또는 생성 설정이면 해당 override를 제거하면 새 UID에 생성합니다.
SALIENCY_INPUT_OVERRIDES = {
    run_id: RESULT_ROOT / "saliency_inputs" / run_id / uid
    for run_id, uid in DEFAULT_SALIENCY_INPUTS.items()
}
SALIENCY_INPUT_SETTINGS = dict(
    device="cuda", reuse_test_saliency=True,
    gradcam_batch_size=4, chunk_size=128, faithfulness_batch_size=32,
    faithfulness_maximum_samples=10000,
    occlusion_fraction=.10, random_repeats=5, seed=8972, bootstrap_repeats=2000,
)


## 완료 run과 재사용 범위

Source run의 완료 상태, full scope, checkpoint UID, codec SHA, ADC test core SHA를 확인합니다. Calibration 점수 재생성 시 기존 threshold를 재현해야 합니다. LFW는 기존 maximize-DIR 규칙을 재현 검사에만 사용하며, 새 보정은 세 데이터셋 모두 calibration fit non-mated로 학습합니다.

FIQA는 데이터셋·정렬·FIQA checkpoint가 같을 때 재사용합니다. Saliency는 dataset/FR당 calibration·test의 원본 top1 gallery target을 한 번 준비한 뒤 동일 cohort의 세 PQ 조건에 출처를 기록하여 연결합니다. 무효 heatmap은 FIQA-only fallback이며, faithfulness 실패는 표에 남기고 비교를 계속합니다. 행 누락·SHA 불일치는 중단합니다.

In [2]:
import pandas as pd
from IPython.display import display
from research.experiments.calibration_matrix import (
    inspect_calibration_matrix, prepare_calibration_matrix,
    run_calibration_matrix, summarize_calibration_matrix,
)

plan = inspect_calibration_matrix(PROJECT_ROOT, MODEL_RUN_MATRIX,
    datasets=DATASETS, models=FR_MODELS, profiles=PQ_PROFILES)
display(plan[["dataset_id", "model", "compression_profile", "source_run_id", "test_rows", "ready"]])
display(plan[["dataset_id", "model", "source_run_dir"]].drop_duplicates())
display(plan[["dataset_id", "model", "source_saliency_target", "needs_origin_top1_saliency"]].drop_duplicates())
print(f"선택된 조건: {len(plan)} / 기대 조건: {len(DATASETS) * len(FR_MODELS) * len(PQ_PROFILES)}")
print("EXECUTE =", EXECUTE)


,dataset_id,model,compression_profile,source_run_id,test_rows,ready
0,lfw,arcface,pq_512_m128_b8,20260902-R001-6c8b08b1,2164,True
1,lfw,arcface,pq_512_m64_b8,20260902-R001-6c8b08b1,2164,True
2,lfw,arcface,pq_512_m32_b8,20260902-R001-6c8b08b1,2164,True
3,rfw_custom,arcface,pq_512_m128_b8,20260903-R001-af875d0f,15063,True
4,rfw_custom,arcface,pq_512_m64_b8,20260903-R001-af875d0f,15063,True
5,rfw_custom,arcface,pq_512_m32_b8,20260903-R001-af875d0f,15063,True
6,survface,arcface,pq_512_m128_b8,20260902-R001-61915edf,182159,True
7,survface,arcface,pq_512_m64_b8,20260902-R001-61915edf,182159,True
8,survface,arcface,pq_512_m32_b8,20260902-R001-61915edf,182159,True
9,lfw,adaface,pq_512_m128_b8,20260830-R001-2c810919,2164,True


,dataset_id,model,source_run_dir
0,lfw,arcface,C:\ronbun\runs\lfw_20260902\20260902-R001-6c8b...
3,rfw_custom,arcface,C:\ronbun\runs\rfw_custom_20260903\20260903-R0...
6,survface,arcface,C:\ronbun\runs\survface_20260902\20260902-R001...
9,lfw,adaface,C:\ronbun\runs\lfw_20260830\20260830-R001-2c81...
12,rfw_custom,adaface,C:\ronbun\runs\rfw_custom_20260830\20260830-R0...
15,survface,adaface,C:\ronbun\runs\survface_20260830\20260830-R001...
18,lfw,magface,C:\ronbun\runs\lfw_20260831\20260831-R001-9372...
21,rfw_custom,magface,C:\ronbun\runs\rfw_custom_20260831\20260831-R0...
24,survface,magface,C:\ronbun\runs\survface_20260831\20260831-R001...
27,lfw,edgeface,C:\ronbun\runs\lfw_20260901\20260901-R001-03c9...


,dataset_id,model,source_saliency_target,needs_origin_top1_saliency
0,lfw,arcface,origin_leave_one_out_identity_cosine,True
3,rfw_custom,arcface,origin_leave_one_out_identity_cosine,True
6,survface,arcface,origin_top1_gallery_cosine,False
9,lfw,adaface,origin_leave_one_out_identity_cosine,True
12,rfw_custom,adaface,origin_leave_one_out_identity_cosine,True
15,survface,adaface,origin_top1_gallery_cosine,False
18,lfw,magface,origin_leave_one_out_identity_cosine,True
21,rfw_custom,magface,origin_leave_one_out_identity_cosine,True
24,survface,magface,origin_top1_gallery_cosine,False
27,lfw,edgeface,origin_leave_one_out_identity_cosine,True


선택된 조건: 36 / 기대 조건: 36
EXECUTE = True


In [3]:
inputs = None
single_receipts = stability_receipts = None
single_report = stability_report = None
if EXECUTE:
    inputs = prepare_calibration_matrix(
        PROJECT_ROOT, plan, variant=FIQA_VARIANT, result_root=RESULT_ROOT,
        saliency_overrides=SALIENCY_INPUT_OVERRIDES,
        input_settings=SALIENCY_INPUT_SETTINGS, fiqa_batch_size=FIQA_BATCH_SIZE,
        progress=lambda event: print(event, flush=True),
    )
    display(inputs[["dataset_id", "model", "compression_profile", "faithfulness_status"]])
else:
    print("준비 완료. 첫 셀 EXECUTE=True 후 Kernel Restart → Run All로 전체 실험을 시작합니다.")


{'stage': 'condition', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8'}
{'stage': 'saliency_regenerate', 'dataset': 'lfw', 'model': 'arcface', 'reason': 'existing target is not split-matched origin-top1'}
{'stage': 'preflight', 'torch': '2.7.1+cu118', 'cuda_runtime': '11.8', 'device': 'cuda', 'device_name': 'NVIDIA GeForce GTX 1080 Ti'}
{'stage': 'source_verified', 'calibration_queries': 3019, 'test_queries': 2164, 'faithfulness_samples': 3019, 'reuse_test_saliency': False}
{'stage': 'reuse_completed', 'directory': 'C:\\ronbun\\results\\calibration\\saliency_inputs\\20260902-R001-6c8b08b1\\saliency-inputs-3e7b614f1006398485edac65'}
{'stage': 'condition', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m64_b8'}
{'stage': 'condition', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m32_b8'}
{'stage': 'condition', 'dataset': 'rfw_custom', 'model': 'arcface', 'profile': 'pq_512_m128_b8'}
{'stage': 'saliency_regenerate', 'dataset': 'rfw_custom', 'model': 

,dataset_id,model,compression_profile,faithfulness_status
0,lfw,arcface,pq_512_m128_b8,passed
1,lfw,arcface,pq_512_m64_b8,passed
2,lfw,arcface,pq_512_m32_b8,passed
3,rfw_custom,arcface,pq_512_m128_b8,passed
4,rfw_custom,arcface,pq_512_m64_b8,passed
5,rfw_custom,arcface,pq_512_m32_b8,passed
6,survface,arcface,pq_512_m128_b8,failed
7,survface,arcface,pq_512_m64_b8,failed
8,survface,arcface,pq_512_m32_b8,failed
9,lfw,adaface,pq_512_m128_b8,failed


## ① 공통 CI

모든 조건에서 동일 seed 8972와 bootstrap 설정으로 Global-safe, FIQA 2/5-bin, Continuous FIQA, +outside, +entropy, +both를 비교합니다. 01과 02의 FIQA baseline 수치가 동일한지 검사합니다. TPIR20은 정답 점수 threshold 통과 및 정답 순위 ≤20을 모두 요구합니다. 실제 FPIR과 목표 충족을 함께 확인하세요.

In [4]:
fit_settings = dict(
    target_fpirs=TARGET_FPIRS, resamples=BOOTSTRAP_RESAMPLES, bootstrap_seed=BOOTSTRAP_SEED,
    safety_fraction=SAFETY_FRACTION, knot_quantiles=KNOT_QUANTILES,
    smoothing=SMOOTHING, ridge=RIDGE, max_iterations=MAX_ITERATIONS,
    minimum_group_non_mated=MINIMUM_GROUP_NON_MATED, shrinkage_strength=SHRINKAGE_STRENGTH,
    progress=lambda event: print(event, flush=True),
)
if EXECUTE:
    single_receipts = run_calibration_matrix(inputs, MATRIX_ROOT / "jobs",
        partition_seeds=(PARTITION_SEED,), **fit_settings)
    single_report = summarize_calibration_matrix(single_receipts, plan,
        expected_seeds=(PARTITION_SEED,), output_root=MATRIX_ROOT / "reports")
    display(single_report["method_summary"])
    print(single_report["directory"])


{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 8972, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 8972, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m64_b8', 'seed': 8972, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m64_b8', 'seed': 8972, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m32_b8', 'seed': 8972, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m32_b8', 'seed': 8972, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'rfw_custom', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 8972, 'reuse': False}
{'stage': 'saliency', 'dataset': 'rfw_custom', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 8972, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'rfw_custom', 'model': 'a

,schema_version,metric_contract,rank_k,model_uid,method,target_fpir,score_space,test_probe_count,test_non_mated_count,test_mated_count,...,source_run_id,family,result_dir,fallback_query_count,fallback_query_fraction,saliency_branch_enabled,faithfulness_status,strong_faithfulness_pass,faithfulness_policy,fallback_policy
0,2,genuine-score-topk-v2,20,arcface-7972a704552df378345f,global_safe,0.01,negative_squared_l2_adc,2164,1688,476,...,20260902-R001-6c8b08b1,fiqa,C:\ronbun\results\calibration\matrix\jobs\job-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,genuine-score-topk-v2,20,arcface-7972a704552df378345f,fiqa_2bin,0.01,negative_squared_l2_adc,2164,1688,476,...,20260902-R001-6c8b08b1,fiqa,C:\ronbun\results\calibration\matrix\jobs\job-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,genuine-score-topk-v2,20,arcface-7972a704552df378345f,fiqa_5bin,0.01,negative_squared_l2_adc,2164,1688,476,...,20260902-R001-6c8b08b1,fiqa,C:\ronbun\results\calibration\matrix\jobs\job-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,genuine-score-topk-v2,20,arcface-7972a704552df378345f,continuous_fiqa,0.01,negative_squared_l2_adc,2164,1688,476,...,20260902-R001-6c8b08b1,fiqa,C:\ronbun\results\calibration\matrix\jobs\job-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,genuine-score-topk-v2,20,arcface-7972a704552df378345f,global_safe,0.05,negative_squared_l2_adc,2164,1688,476,...,20260902-R001-6c8b08b1,fiqa,C:\ronbun\results\calibration\matrix\jobs\job-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1255,2,genuine-score-topk-v2,20,edgeface-a348c305af33c223b337,plus_entropy,0.20,negative_squared_l2_adc,182159,121736,60423,...,20260901-R001-56c2f3ed,saliency,C:\ronbun\results\calibration\matrix\jobs\job-...,149.0,0.000818,True,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
1256,2,genuine-score-topk-v2,20,edgeface-a348c305af33c223b337,plus_both,0.20,negative_squared_l2_adc,182159,121736,60423,...,20260901-R001-56c2f3ed,saliency,C:\ronbun\results\calibration\matrix\jobs\job-...,149.0,0.000818,True,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
1257,2,genuine-score-topk-v2,20,edgeface-a348c305af33c223b337,plus_outside,0.30,negative_squared_l2_adc,182159,121736,60423,...,20260901-R001-56c2f3ed,saliency,C:\ronbun\results\calibration\matrix\jobs\job-...,149.0,0.000818,True,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
1258,2,genuine-score-topk-v2,20,edgeface-a348c305af33c223b337,plus_entropy,0.30,negative_squared_l2_adc,182159,121736,60423,...,20260901-R001-56c2f3ed,saliency,C:\ronbun\results\calibration\matrix\jobs\job-...,149.0,0.000818,True,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1


C:\ronbun\results\calibration\matrix\reports\matrix-report-2e2ab02ddb1d039ef1084383


## ② 분할 안정성

사전 지정한 0–18, 8972의 전체 20개 calibration fit/safety 분할을 평가합니다. 같은 test/gallery를 공유하며 독립 데이터셋 반복이 아닙니다. seed별 완료 결과를 저장하므로 중단 후 동일 설정으로 Run All 하면 완료된 fit을 건너뜁니다. seed 8972는 앞 단계 결과를 재사용합니다.

In [5]:
if EXECUTE and RUN_SPLIT_STABILITY:
    stability_receipts = run_calibration_matrix(inputs, MATRIX_ROOT / "jobs",
        partition_seeds=SPLIT_SEEDS, **fit_settings)
    stability_report = summarize_calibration_matrix(stability_receipts, plan,
        expected_seeds=SPLIT_SEEDS, output_root=MATRIX_ROOT / "reports")
    display(stability_report["split_summary"])
    print(stability_report["directory"])


{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 0, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 0, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 1, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 1, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 2, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 2, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 3, 'reuse': False}
{'stage': 'saliency', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'seed': 3, 'reuse': False}
{'stage': 'fiqa', 'dataset': 'lfw', 'model': 'arcface', 'profile': 'pq_512_m128_b8', 'se

PermissionError: [WinError 5] 액세스가 거부되었습니다: 'C:\\ronbun\\results\\calibration\\matrix\\jobs\\job-6a677d8bd1407c62d136af57\\.staging-df68b6846f3541abb76b659d68d47399' -> 'C:\\ronbun\\results\\calibration\\matrix\\jobs\\job-6a677d8bd1407c62d136af57\\saliency-incremental-7dff2a13afdb279d8f66426c'

## ③ 보고 통합

데이터셋·FR·PQ별 결과와 paired CI를 보존합니다. 36조건/20seed의 전체 완료 여부를 별도로 표시하며 평균으로 데이터셋을 합치지 않습니다. Saliency paired 차이의 reference는 Continuous FIQA입니다. 압축률별 성능은 별도 행으로 비교하며 압축률 간 차이의 CI로 해석하지 않습니다.

FPIR 목표 미달, 고정 threshold에 대한 CI, 다중 비교 미보정, offline 원본 gallery 사용, RFW의 checkpoint 학습 identity 중복 미확인(UNKNOWN)을 해석에 유지하세요. 개선 여부에 따라 조건을 제외하지 않습니다.

In [ ]:
if EXECUTE:
    report = stability_report if stability_report is not None else single_report
    display(pd.DataFrame([{
        "status": report["manifest"]["status"],
        "expected_jobs": report["manifest"]["expected_jobs"],
        "observed_jobs": report["manifest"]["observed_jobs"],
        "full_split_panel": stability_report is not None,
        "artifact": str(report["directory"]),
    }]))
    display(report["split_summary"])
    display(report["split_summary"].pivot(index=["dataset_id", "model", "method", "target_fpir"],
        columns="compression_profile", values=["tpir_median", "fpir_median", "target_met_split_count"]))
    display(report["paired_comparisons"])
    print("CSV 및 manifest:", report["directory"])
else:
    print("사전 검증만 실행했습니다. 정식 성능 결과는 아직 생성하지 않았습니다.")
